In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import scipy.optimize

import mrfitty
from mrfitty.base import (
    AdaptiveEnergyRangeBuilder,
    InterpolatedReferenceSpectraSet,
    ReferenceSpectrum,
)

%matplotlib inline

In [ ]:
import fnmatch


def filter_spectra_by_name(spectra_list, *patterns):
    """Return spectra whose file_name matches any of the glob-style patterns."""
    matches = [s for s in spectra_list if any(fnmatch.fnmatch(s.file_name, p) for p in patterns)]
    if not matches:
        raise ValueError(
            f"No spectra matched the given pattern(s): {patterns!r}. "
            f"Available file names: {[s.file_name for s in spectra_list]}"
        )
    return matches


In [ ]:
src_path, _ = os.path.split(mrfitty.__path__[0])
sample_data_dir_path = os.path.join(src_path, 'example', 'arsenic')
print('sample data is installed at "{}"'.format(sample_data_dir_path))
os.path.exists(sample_data_dir_path)

In [ ]:
sample_data_reference_glob = os.path.join(sample_data_dir_path, 'reference/*.e')
print('sample data reference glob: {}'.format(sample_data_reference_glob))
sample_data_unknown_glob = os.path.join(sample_data_dir_path, 'unknown/*.e')
print('sample data unknown glob: {}'.format(sample_data_unknown_glob))

In [ ]:
sample_data_reference_set, _ = list(ReferenceSpectrum.read_all([sample_data_reference_glob]))
sample_data_reference_list = sorted(list(sample_data_reference_set), key=lambda s: s.file_name)
print('sample data reference file count: {}'.format(len(sample_data_reference_list)))
sample_data_unknown_set, _ = list(ReferenceSpectrum.read_all([sample_data_unknown_glob]))
sample_data_unknown_list = sorted(list(sample_data_unknown_set), key=lambda s: s.file_name)
print('sample data unknown file count: {}'.format(len(sample_data_unknown_list)))

In [ ]:
unknown_spectrum = filter_spectra_by_name(sample_data_unknown_list, "OTT3_55*")[0]
print(f'unknown spectrum: {unknown_spectrum.file_name}')
reference_spectra = filter_spectra_by_name(
    sample_data_reference_list,
    "Arsenopyrite_Jul*", "orpiment_all*", "arsenate*_diop*") 

interp_ref_set = InterpolatedReferenceSpectraSet(
    unknown_spectrum=unknown_spectrum,
    reference_set=reference_spectra,
)

subset = interp_ref_set.get_reference_subset_and_unknown_df(
    reference_list=reference_spectra,
    energy_range_builder=AdaptiveEnergyRangeBuilder(),
)

A = subset['reference_subset_df'].values
b = subset['unknown_subset_df']['norm'].values
energies = subset['reference_subset_df'].index.values
ref_names = list(subset['reference_subset_df'].columns)

coef, _ = scipy.optimize.nnls(A, b)
fitted = A @ coef

print('Coefficients:')
for ref_coef, ref_name in sorted(zip(coef, ref_names), reverse=True):
    print(f"{ref_coef:0.6f}: {ref_name}")

In [ ]:
residuals = fitted - b
rmse = np.sqrt(np.mean(np.square(residuals)))
print(f'Residuals: n={len(residuals)}, mean={residuals.mean():.6f}, std={residuals.std():.6f}')
print(f'RMSE: {rmse:.6f}')

In [ ]:
fig, ax_fit = plt.subplots(nrows=1, ncols=1, figsize=(10, 4), sharex=True)

ax_fit.plot(energies, b, label='unknown', color='black')
ax_fit.plot(energies, fitted, label='fit', color='red', linestyle='--')
ax_fit.scatter(energies, residuals, color='orange', label='residuals', marker='o', s=10.0)
ax_fit.axhline(0, color='black', linewidth=0.5)
ax_fit.set_xlabel('Energy (eV)')
ax_fit.set_ylabel('Normalized Fluorescence')
ax_fit.legend()
ax_fit.set_title(f'NNLS Fit: {unknown_spectrum.file_name}')

plt.tight_layout()
plt.show()

In [ ]:
n = len(residuals)
n_lags = min(40, n // 2)
x = residuals - residuals.mean()
var = np.dot(x, x) / n
acf_values = np.array(
    [1.0] + [np.dot(x[:-lag], x[lag:]) / (n * var) for lag in range(1, n_lags + 1)]
)
lags = np.arange(n_lags + 1)

print(f'ACF at lag 0: {acf_values[0]:.4f}')
print(f'ACF at lag 1: {acf_values[1]:.4f}')
print(f'ACF at lag 2: {acf_values[2]:.4f}')
print(f'ACF at lag 5: {acf_values[5]:.4f}')

In [ ]:
ci_95 = 1.96 / np.sqrt(n)

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(lags, acf_values, color='steelblue', alpha=0.7)
ax.axhline(ci_95, color='red', linestyle='--', label='95% CI (white noise)')
ax.axhline(-ci_95, color='red', linestyle='--')
ax.axhline(0, color='black', linewidth=0.5)
ax.set_xlabel('Lag')
ax.set_ylabel('Autocorrelation')
ax.set_title('Autocorrelation Function of Residuals')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
rng = np.random.default_rng(seed=42)
n_bootstrap = 1000
block_length = max(1, int(np.round(n ** (1 / 3))))
n_blocks_needed = int(np.ceil(n / block_length))
n_full_blocks = n // block_length
n_holdout_blocks = round(n_full_blocks / 3)
# assume n=198 and block_length=6
# then block_starts looks like
#  array([  0,   1,   2, ..., 192 ])
block_starts = np.arange(n - block_length + 1)

print(f'n={n}, block_length={block_length}, blocks per sample={n_blocks_needed}')
print(f'n_full_blocks={n_full_blocks}, n_holdout_blocks={n_holdout_blocks} (~{n_holdout_blocks * block_length / n:.0%} of data)')

bootstrap_coefs = np.zeros((n_bootstrap, len(coef)))
bootstrap_pes = np.zeros(n_bootstrap)

for i in range(n_bootstrap):
    # Randomly select non-contiguous holdout blocks totaling ~1/3 of the data
    # holdout_block_indices look like
    #  array([ 6, 20,  8,  3, 16, 28, 24,  2, 32, 22, 18])
    holdout_block_indices = rng.choice(n_full_blocks, size=n_holdout_blocks, replace=False)
    holdout_mask = np.zeros(n, dtype=bool)
    for idx in holdout_block_indices:
        holdout_mask[idx * block_length:(idx + 1) * block_length] = True
    train_mask = ~holdout_mask

    # Restrict bootstrap block starts to positions that don't overlap any holdout block
    valid_block_starts = block_starts[
        np.array([not holdout_mask[s:s + block_length].any() for s in block_starts])
    ]

    # Build bootstrap sample from moving blocks of residuals (holdout positions excluded)
    sampled_starts = rng.choice(valid_block_starts, size=n_blocks_needed, replace=True)
    bootstrap_residuals = np.concatenate(
        [residuals[s:s + block_length] for s in sampled_starts]
    )[:n]
    bootstrap_b = fitted + bootstrap_residuals

    # Fit on bootstrap data with holdout blocks removed
    bootstrap_coef, _ = scipy.optimize.nnls(A[train_mask], bootstrap_b[train_mask])
    bootstrap_coefs[i] = bootstrap_coef

    # Prediction error on real data at the held-out positions
    holdout_residuals = A[holdout_mask] @ bootstrap_coef - b[holdout_mask]
    bootstrap_pes[i] = np.sqrt(np.mean(np.square(holdout_residuals)))

print('Bootstrap complete')
print(f'Coefficient means: {bootstrap_coefs.mean(axis=0)}')
print(f'Coefficient stds:  {bootstrap_coefs.std(axis=0)}')
print(f'Prediction error mean={bootstrap_pes.mean():.6f}, std={bootstrap_pes.std():.6f}')


In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))

ax.plot(energies, b, label='unknown spectrum', color='black', linewidth=1.0)
ax.scatter(
    energies[~holdout_mask], b[~holdout_mask],
    color='steelblue', s=10, zorder=4, alpha=0.6,
    label=f'training ({(~holdout_mask).sum()} points)',
)
ax.scatter(
    energies[holdout_mask], b[holdout_mask],
    color='orange', s=20, zorder=5,
    label=f'holdout ({holdout_mask.sum()} points)',
)

ax.set_xlabel('Energy (eV)')
ax.set_ylabel('Normalized Fluorescence')
ax.set_title(
    f'Last Holdout Mask — {unknown_spectrum.file_name}\n'
    f'block_length={block_length}, n_holdout_blocks={n_holdout_blocks}'
)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.gridspec as gridspec

n_refs = len(ref_names)
n_cols = n_refs + 1

fig = plt.figure(figsize=(4 * n_cols, 16))
gs = gridspec.GridSpec(4, n_cols, figure=fig)

ax_fit = fig.add_subplot(gs[0, :])
ax_acf = fig.add_subplot(gs[1, :])
axs = np.array([[fig.add_subplot(gs[row, col]) for col in range(n_cols)] for row in range(2, 4)])

rmse = np.sqrt(np.mean(residuals ** 2))

# Row 0: fit plot
ax_fit.plot(energies, b, label=unknown_spectrum.file_name, color='black')
ax_fit.plot(energies, fitted, label='fit', color='red', linestyle='--')
ax_fit.scatter(energies, residuals, color='orange', label=f'residuals (RMSE={rmse:.4f})', marker='o', s=10.0)
ax_fit.axhline(0, color='black', linewidth=0.5)
ax_fit.set_xlabel('Energy (eV)')
ax_fit.set_ylabel('Normalized Fluorescence')
ax_fit.legend()
ax_fit.set_title(f'{unknown_spectrum.file_name}')

# Row 1: residual autocorrelation
ax_acf.bar(lags, acf_values, color='steelblue', alpha=0.7)
ax_acf.axhline(ci_95, color='red', linestyle='--', label='95% CI (white noise)')
ax_acf.axhline(-ci_95, color='red', linestyle='--')
ax_acf.axhline(0, color='black', linewidth=0.5)
ax_acf.set_xlabel('Lag')
ax_acf.set_ylabel('Autocorrelation')
ax_acf.set_title('Residual Autocorrelation Function')
ax_acf.legend(fontsize=8)

# Rows 2–3: bootstrap coefficient and prediction error distributions
for j, (coef_mean, coef_i, name) \
    in enumerate(sorted(zip(bootstrap_coefs.mean(axis=0), range(n_refs), ref_names), reverse=True)):

    p2_5 = np.percentile(bootstrap_coefs[:, coef_i], 2.5)
    p97_5 = np.percentile(bootstrap_coefs[:, coef_i], 97.5)

    axs[0, j].hist(bootstrap_coefs[:, coef_i], bins=40, color='steelblue', alpha=0.7, edgecolor='white')
    axs[0, j].axvline(coef[coef_i], color='red', linestyle='--', label=f'observed={coef[coef_i]:.3f}')
    axs[0, j].axvline(coef_mean, color='blue', linestyle='--', label=f'mean={coef_mean:.3f}')
    axs[0, j].axvline(p2_5, color='green', linestyle=':', label=f'2.5%={p2_5:.3f}')
    axs[0, j].axvline(p97_5, color='green', linestyle=':', label=f'97.5%={p97_5:.3f}')
    axs[0, j].set_title(name, fontsize=9)
    axs[0, j].set_xlabel('Coefficient')
    axs[0, j].legend(fontsize=8)

    axs[1, j].violinplot(bootstrap_coefs[:, coef_i])
    axs[1, j].scatter(
        [0.95], [coef[coef_i]], color='red', zorder=5, marker='o', s=60,
        edgecolors='black', linewidths=0.8, label=f'observed={coef[coef_i]:.3f}',
    )
    axs[1, j].scatter(
        [1.05], [coef_mean], color='blue', zorder=5, marker='D', s=60,
        edgecolors='black', linewidths=0.8, label=f'mean={coef_mean:.3f}',
    )
    axs[1, j].set_title(name, fontsize=9)
    axs[1, j].legend(fontsize=8)

pes_mean = bootstrap_pes.mean()
pes_p2_5 = np.percentile(bootstrap_pes, 2.5)
pes_p97_5 = np.percentile(bootstrap_pes, 97.5)

axs[0, n_refs].hist(bootstrap_pes, bins=40, color='darkorange', alpha=0.7, edgecolor='white')
axs[0, n_refs].axvline(rmse, color='red', linestyle='--', label=f'RMSE={rmse:.4f}')
axs[0, n_refs].axvline(pes_mean, color='blue', linestyle='--', label=f'mean={pes_mean:.4f}')
axs[0, n_refs].axvline(pes_p2_5, color='green', linestyle=':', label=f'2.5%={pes_p2_5:.4f}')
axs[0, n_refs].axvline(pes_p97_5, color='green', linestyle=':', label=f'97.5%={pes_p97_5:.4f}')
axs[0, n_refs].set_title('Holdout Prediction Error')
axs[0, n_refs].set_xlabel('Prediction Error (RMSE)')
axs[0, n_refs].legend(fontsize=8)

parts = axs[1, n_refs].violinplot(bootstrap_pes)
for pc in parts['bodies']:
    pc.set_facecolor('darkorange')
    pc.set_alpha(0.7)
axs[1, n_refs].scatter(
    [0.95], [rmse], color='red', zorder=5, marker='o', s=60,
    edgecolors='black', linewidths=0.8, label=f'RMSE={rmse:.4f}',
)
axs[1, n_refs].scatter(
    [1.05], [pes_mean], color='blue', zorder=5, marker='D', s=60,
    edgecolors='black', linewidths=0.8, label=f'mean={pes_mean:.4f}',
)
axs[1, n_refs].set_title('Holdout Prediction Error')
axs[1, n_refs].legend(fontsize=8)

# Synchronize y-axis range across coefficient violin plots, excluding prediction error
coef_violin_axes = axs[1, :n_refs]
all_ylims = [ax.get_ylim() for ax in coef_violin_axes]
global_ymin = min(lo for lo, hi in all_ylims)
global_ymax = max(hi for lo, hi in all_ylims)
for ax in coef_violin_axes:
    ax.set_ylim(global_ymin, global_ymax)

plt.suptitle(f'Moving Block Holdout Bootstrap Distributions ({n_bootstrap} iterations)', fontsize=13)
plt.tight_layout()
plt.show()